<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import os
from dotenv import load_dotenv
from collections import OrderedDict

# imports
import requests
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
import indicators
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

load_dotenv()

ModuleNotFoundError: No module named 'yfinance'

In [ ]:
stocks = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA", "NFLX", "AMD", "INTC",
    "JPM", "GS", "BAC", "C", "WFC", "V", "MA", "AXP", "BRK-B",
    "UNH", "JNJ", "PFE", "LLY", "ABBV", "TMO", "DHR", "BMY", "GILD",
    "XOM", "CVX", "COP", "OXY", "SLB", "HAL", "BP", "SHEL", "EOG",
    "WMT", "COST", "HD", "LOW", "MCD", "SBUX", "TGT", "NKE", "PG", "KO",
    "ADBE", "CRM", "ORCL", "QCOM", "AVGO", "TXN", "INTU", "CSCO", "IBM", "MU",
    "PANW", "NOW", "CDNS", "ANET", "LRCX", "SNPS", "ACN", "FTNT", "MSI", "ZM",
    "BLK", "TFC", "USB", "BK", "SCHW", "SPGI", "MS", "ICE", "PGR", "AIG",
    "CB", "MMC", "MET", "ALL", "PRU", "CME", "COF", "DFS", "FITB", "MTB",
    "ABT", "MRK", "ZBH", "ISRG", "MDT", "CVS", "CI", "REGN", "VRTX", "SYK",
    "PSX", "MPC", "VLO", "KMI", "WMB", "CTRA", "DVN", "HES", "FANG",
    "LIN", "APD", "ECL", "SHW", "NEM", "FCX", "DD", "ALB", "LYB", "MLM",
    "PEP", "CL", "KMB", "EL", "MNST", "KR", "DG", "DLTR", "GIS", "MDLZ",
    "TSCO", "ROST", "TJX", "YUM", "WBA", "PM", "MO", "UL", "HSY", "HRL",
    "CAT", "DE", "HON", "LMT", "RTX", "BA", "GE", "NOC", "ETN", "EMR",
    "UNP", "NSC", "FDX", "UPS", "CSX", "WM", "EXC", "DUK", "NEE",
    "PLD", "AMT", "CCI", "EQIX", "SPG", "PSA", "O", "DLR", "VTR", "ARE"
]

# The training set uses all stocks from the stocks list
overall_tickers = stocks

# Alternatively, use stocks2 if you want to
stocks2 = [
    "ADBE", "CRM", "ORCL", "SAP", "NOW", "SHOP", "SQ", "ZM", "CRWD", "DDOG",
    "TXN", "QCOM", "AVGO", "MU", "LRCX", "KLAC", "NXPI", "ADI", "MRVL", "SWKS",
    "PYPL", "INTU", "FISV", "ADP", "VEEV", "TEAM", "WDAY", "ZS", "OKTA", "MDB",
    "T", "VZ", "TMUS", "CHTR", "CMCSA", "DIS", "ROKU", "LYV", "TTWO", "ATVI",
    "PEP", "KMB", "CL", "HSY", "MDLZ", "GIS", "MO", "PM", "EL", "STZ"
]

In [ ]:
API_KEY = os.environ["FRED_API_KEY"]

def get_fred_data(measures, API_KEY):
    ret = None
    for measure in measures:
        params = {
            "series_id": measure,
            "api_key": API_KEY,
            "file_type": "json",
            "observation_start": "1999-01-01",
            "observation_end": pd.Timestamp.now().strftime("%Y-%m-%d"),
        }
        url = "https://api.stlouisfed.org/fred/series/observations"
        response = requests.get(url, params=params).json()

        records = []
        for obs in response["observations"]:
            value = obs["value"]
            value = float(value) if value != "." else None
            records.append({"date": obs["date"], measure: value})

        df = pd.DataFrame(records)
        df["date"] = pd.to_datetime(df["date"])
        if ret is None:
            ret = df
        else:
            ret = ret.merge(df, on="date", how="outer")

    return ret

# FRED series:
#   PCEPI / FEDFUNDS / UNRATE  - inflation, rates, unemployment (monthly)
#   UMCSENT                    - consumer sentiment (monthly)
#   GDPC1                      - real GDP (quarterly)
#   WALCL                      - Fed balance sheet (weekly, $M)
#   VIXCLS                     - VIX close (daily)
#   T10Y2Y                     - term spread, 10Y minus 2Y treasury (daily)
#   BAA10Y                     - credit spread, Baa minus 10Y treasury (daily)
#   DTWEXBGS                   - trade-weighted dollar index (daily)
fred_data = get_fred_data(
    ["PCEPI", "FEDFUNDS", "UNRATE", "UMCSENT", "GDPC1", "WALCL",
     "VIXCLS", "T10Y2Y", "BAA10Y", "DTWEXBGS"],
    API_KEY,
)
fred_data["real_pce_yoy"] = fred_data["PCEPI"].pct_change(12) * 100
fred_data = (
    fred_data.set_index("date")
             .drop(columns=["PCEPI"])
             .sort_index()
             .ffill()
             .dropna(how="all")
)


In [ ]:
import time
from tqdm import tqdm

def fetch_historical_fundamentals(tickers, delay=0.5):
    """
    Fetch quarterly income statement, balance sheet, and cash flow data.
    Returns a dict of {ticker: DataFrame} with quarterly dates as index.
    """
    historical_fund = {}
    failed = []

    for ticker_str in tqdm(tickers, desc="Fetching historical financials"):
        try:
            t = yf.Ticker(ticker_str)
            inc = t.quarterly_financials
            bs = t.quarterly_balance_sheet
            cf = t.quarterly_cashflow

            if inc is None or inc.empty:
                failed.append(ticker_str)
                continue

            quarters = inc.columns
            records = []
            for q in quarters:
                row = {"Date": q, "Stock": ticker_str}

                if "Net Income" in inc.index:
                    row["NetIncome"] = inc.loc["Net Income", q]
                if "Total Revenue" in inc.index:
                    row["Revenue"] = inc.loc["Total Revenue", q]
                if "Operating Income" in inc.index:
                    row["OperatingIncome"] = inc.loc["Operating Income", q]
                if "EBITDA" in inc.index:
                    row["EBITDA"] = inc.loc["EBITDA", q]

                if bs is not None and not bs.empty:
                    if "Total Assets" in bs.index and q in bs.columns:
                        row["TotalAssets"] = bs.loc["Total Assets", q]
                    if "Stockholders Equity" in bs.index and q in bs.columns:
                        row["Equity"] = bs.loc["Stockholders Equity", q]
                    if "Total Debt" in bs.index and q in bs.columns:
                        row["TotalDebt"] = bs.loc["Total Debt", q]
                    if "Ordinary Shares Number" in bs.index and q in bs.columns:
                        row["SharesOutstanding"] = bs.loc["Ordinary Shares Number", q]
                    if "Current Assets" in bs.index and q in bs.columns:
                        row["CurrentAssets"] = bs.loc["Current Assets", q]
                    if "Current Liabilities" in bs.index and q in bs.columns:
                        row["CurrentLiabilities"] = bs.loc["Current Liabilities", q]

                if cf is not None and not cf.empty:
                    if "Free Cash Flow" in cf.index and q in cf.columns:
                        row["FreeCashFlow"] = cf.loc["Free Cash Flow", q]
                    if "Operating Cash Flow" in cf.index and q in cf.columns:
                        row["OperatingCashFlow"] = cf.loc["Operating Cash Flow", q]

                records.append(row)

            if records:
                historical_fund[ticker_str] = pd.DataFrame(records).set_index("Date").sort_index()
        except Exception as e:
            print(f"  Failed for {ticker_str}: {e}")
            failed.append(ticker_str)
        time.sleep(delay)

    if failed:
        print(f"\nFailed tickers ({len(failed)}): {failed}")
    return historical_fund


In [ ]:
def compute_historical_fundamentals(historical_fund_dict, stock_data_filename):
    """
    Derive point-in-time fundamental ratios from quarterly financials.
    Computes TTM (trailing 4 quarter) metrics, merges with weekly close
    prices, and forward-fills to weekly frequency. No look-ahead bias.

    Returns a dict of {ticker: DataFrame} indexed on weekly dates with columns:
      HistPE, PB, ProfitMargin, ROE, ROA, DebtToEquity, FCFYield,
      RevenueGrowthYoY, EarningsGrowthYoY, PE_Change_4wk,
      EPS, FCF, EBITDA_TTM, OperatingIncome_TTM, ROIC, CurrentRatio, PEG
    """
    fund_dict = {}

    for ticker, qdf in historical_fund_dict.items():
        qdf = qdf.copy()

        # --- TTM (trailing 4 quarter) aggregations ---
        for raw_col, ttm_col in [
            ("NetIncome",       "NetIncome_TTM"),
            ("Revenue",         "Revenue_TTM"),
            ("FreeCashFlow",    "FCF_TTM"),
            ("EBITDA",          "EBITDA_TTM"),
            ("OperatingIncome", "OperatingIncome_TTM"),
        ]:
            if raw_col in qdf.columns:
                qdf[ttm_col] = qdf[raw_col].rolling(4, min_periods=4).sum()

        # --- YoY growth (compare to 4 quarters ago) ---
        if "Revenue_TTM" in qdf.columns:
            qdf["RevenueGrowthYoY"] = qdf["Revenue_TTM"].pct_change(periods=4)
        if "NetIncome_TTM" in qdf.columns:
            qdf["EarningsGrowthYoY"] = qdf["NetIncome_TTM"].pct_change(periods=4)

        # --- Point-in-time balance sheet ratios ---
        if "Equity" in qdf.columns and "TotalDebt" in qdf.columns:
            qdf["DebtToEquity"] = qdf["TotalDebt"] / qdf["Equity"].replace(0, np.nan)
        if "NetIncome_TTM" in qdf.columns and "Equity" in qdf.columns:
            qdf["ROE"] = qdf["NetIncome_TTM"] / qdf["Equity"].replace(0, np.nan)
        if "NetIncome_TTM" in qdf.columns and "TotalAssets" in qdf.columns:
            qdf["ROA"] = qdf["NetIncome_TTM"] / qdf["TotalAssets"].replace(0, np.nan)
        if "NetIncome_TTM" in qdf.columns and "Revenue_TTM" in qdf.columns:
            qdf["ProfitMargin"] = qdf["NetIncome_TTM"] / qdf["Revenue_TTM"].replace(0, np.nan)

        # --- ROIC = OperatingIncome_TTM / (TotalDebt + Equity) ---
        if all(c in qdf.columns for c in ["OperatingIncome_TTM", "TotalDebt", "Equity"]):
            invested_capital = (qdf["TotalDebt"] + qdf["Equity"]).replace(0, np.nan)
            qdf["ROIC"] = qdf["OperatingIncome_TTM"] / invested_capital

        # --- Current Ratio ---
        if "CurrentAssets" in qdf.columns and "CurrentLiabilities" in qdf.columns:
            qdf["CurrentRatio"] = qdf["CurrentAssets"] / qdf["CurrentLiabilities"].replace(0, np.nan)

        # --- EPS for P/E ---
        if "NetIncome_TTM" in qdf.columns and "SharesOutstanding" in qdf.columns:
            qdf["EPS_TTM"] = qdf["NetIncome_TTM"] / qdf["SharesOutstanding"].replace(0, np.nan)

        # --- Merge with weekly prices for price-based ratios ---
        try:
            prices = extract_ticker_dataframe(stock_data_filename, ticker)
            weekly_close = prices["Close"]
        except (ValueError, KeyError):
            continue

        weekly_q = qdf.resample("W-FRI").ffill()
        weekly_q = weekly_q.reindex(weekly_close.index, method="ffill")

        result = pd.DataFrame(index=weekly_close.index)

        # Price-based ratios + EPS passthrough
        if "EPS_TTM" in weekly_q.columns:
            result["EPS"] = weekly_q["EPS_TTM"]
            result["HistPE"] = weekly_close / weekly_q["EPS_TTM"].replace(0, np.nan)
            result["PE_Change_4wk"] = result["HistPE"].pct_change(periods=4)

        if "Equity" in weekly_q.columns and "SharesOutstanding" in weekly_q.columns:
            book_per_share = weekly_q["Equity"] / weekly_q["SharesOutstanding"].replace(0, np.nan)
            result["PB"] = weekly_close / book_per_share.replace(0, np.nan)

        if "FCF_TTM" in weekly_q.columns:
            result["FCF"] = weekly_q["FCF_TTM"]
            if "SharesOutstanding" in weekly_q.columns:
                market_cap = weekly_close * weekly_q["SharesOutstanding"]
                result["FCFYield"] = weekly_q["FCF_TTM"] / market_cap.replace(0, np.nan)

        # PEG = PE / (EarningsGrowthYoY as percent)
        if "HistPE" in result.columns and "EarningsGrowthYoY" in weekly_q.columns:
            growth_pct = weekly_q["EarningsGrowthYoY"] * 100
            result["PEG"] = result["HistPE"] / growth_pct.replace(0, np.nan)

        # Direct passthroughs from weekly_q
        for col in ["ProfitMargin", "ROE", "ROA", "DebtToEquity",
                    "RevenueGrowthYoY", "EarningsGrowthYoY",
                    "EBITDA_TTM", "OperatingIncome_TTM", "ROIC", "CurrentRatio"]:
            if col in weekly_q.columns:
                result[col] = weekly_q[col]

        result = result.replace([np.inf, -np.inf], np.nan)
        fund_dict[ticker] = result

    return fund_dict


In [ ]:
# Fetch and cache historical quarterly financials
historical_fundamentals = fetch_historical_fundamentals(overall_tickers, delay=0.5)

# Save combined long-format DataFrame for caching
hist_fund_frames = []
for ticker, df in historical_fundamentals.items():
    hist_fund_frames.append(df)
hist_fund_all = pd.concat(hist_fund_frames).reset_index()
hist_fund_all.to_csv("data/historical_fundamentals.csv", index=False)
print(f"Historical fundamentals shape: {hist_fund_all.shape}")
print(f"Tickers covered: {hist_fund_all['Stock'].nunique()}")
hist_fund_all.head()

In [ ]:
def download_and_fix_yfinance_data(stocks, csv_filename): # Takes raw yfinance dataframe and fixes it if it has that weird date syncing problem
    data = {}

    for t in stocks:
        df = yf.download(t, interval='1d', start='2015-01-01', auto_adjust=True)
        df = df.resample('W-FRI').agg({ # everything needs to be on the friday interval
          ('Open', t): 'first',
          ('High', t): 'max',
          ('Low', t): 'min',
          ('Close', t): 'last',
          ('Volume', t): 'sum'
        })
        df.name = t
        data[t] = df

    # Combine and align on the same date index
    combined = pd.concat(data.values(), axis=1, join='outer')
    combined = combined.ffill()  # or .bfill(), depending on your use case

    non_nan_counts = combined.count()
    cols_to_drop = non_nan_counts[non_nan_counts < 20].index
    y = combined.drop(columns=cols_to_drop)

    stockData = y.copy()

    # Rename columns, joining multi-level column names into a single string with "_".
    stockData.columns = ["_".join(col) if isinstance(col, tuple) else col for col in stockData.columns]

    # Remove unnecessary columns such as "level_0" or "index" that may have been carried over.
    stockData = stockData.loc[:, ~stockData.columns.isin(["level_0", "index"])]
    stockData.to_csv(csv_filename + ".csv")

#for sector_list, sector_name in sectors_dict.items():
#  download_and_fix_yfinance_data(sector_list, sector_name)


In [ ]:
def download_and_fix_sp500():
    y = yf.download("^GSPC", start="2015-01-01", interval="1d")
    y = y.resample('W-FRI').agg({
          ('Open', "^GSPC"): 'first',
          ('High', "^GSPC"): 'max',
          ('Low', "^GSPC"): 'min',
          ('Close', "^GSPC"): 'last',
          ('Volume', "^GSPC"): 'sum'
        })
    sp500 = y.copy()

    sp500.columns = ["_".join(col) if isinstance(col, tuple) else col for col in sp500.columns]

    sp500 = sp500.loc[:, ~sp500.columns.isin(["level_0", "index"])]
    sp500.to_csv("SP500.csv")

In [ ]:
def extract_ticker_dataframe(csv_filepath: str, ticker: str) -> pd.DataFrame:
    """
    Reads a multi-ticker CSV file (like one from yfinance) and isolates the data
    for the specified ticker. This updated version assumes that the CSV contains the
    dates as the index (first column), so we use that as the Date information.

    The CSV is expected to have a two-row header:
      - The first row contains field names (e.g., "Open", "High", etc.).
      - The second row contains the ticker symbols for each column.

    The returned DataFrame's columns will be ordered as needed for Backtrader:
      Date, Open, High, Low, Close, Volume.
    """
    # Use the first column as the index so that the dates are read from the CSV.
    df = pd.read_csv(csv_filepath)#, header=[0, 1], index_col=0)

    # Convert the index to datetime in case it's not already
    df.index = pd.to_datetime(df["Date"], errors="coerce")
    df = df.drop("Date", axis=1)

    # Identify all columns for the specified ticker (matching on the lower level).
    ticker_cols = [col for col in df.columns if ticker == str(col).strip().split("_")[-1]]
    if not ticker_cols:
        raise ValueError(f"Ticker '{ticker}' not found in the CSV file.")

    # Extract the ticker's columns
    df_ticker = df.loc[:, ticker_cols].copy()
    #df_ticker.columns = [col[0].strip() for col in df_ticker.columns]

    # Removing ticker name from column names
    for col in df_ticker.columns:
        df_ticker.rename(columns={col: str(col).split("_")[0]}, inplace=True)

    desired_order = ["Open", "High", "Low", "Close", "Volume"]

    available_order = [col for col in desired_order if col in df_ticker.columns]
    df_ticker = df_ticker[available_order]

    df_ticker = df_ticker.reset_index().rename(columns={"index": "Date"}).set_index("Date")

    return df_ticker

In [ ]:
# 1. Feature engineering: Hamilton Regime Switching Model
sp500 = pd.read_csv("data/SP500.csv")
sp500.set_index("Date", inplace=True)
sp500.index = pd.to_datetime(sp500.index)
sp500['Log_Returns'] = np.log(sp500['Close_^GSPC'] / sp500['Close_^GSPC'].shift(1))
sp500['Returns_6wk'] = sp500['Close_^GSPC'].pct_change(6)
sp500 = sp500.dropna()

def classify_regimes(sp500: pd.DataFrame) -> pd.DataFrame:
    model = MarkovRegression(sp500['Log_Returns'], k_regimes=2, trend='c', switching_variance=True)
    result = model.fit()
    #print(result.summary())
    smoothed_probs = result.smoothed_marginal_probabilities
    sp500['Regime'] = smoothed_probs.idxmax(axis=1)
    sp500['Bull_Prob'] = smoothed_probs[0]

    return sp500["Bull_Prob"].to_frame()
    # 0 -> Bull, 1 -> Bear

def compute_beta(asset_returns, market_returns, window=20):
    cov = asset_returns.rolling(window).cov(market_returns)
    var = market_returns.rolling(window).var()
    return cov / var


In [ ]:
# Compute rolling portfolio weights using a lookback period (e.g., 52 weeks)
def compute_rolling_portfolio_weights(data, lookback_window=52):
    """
    Computes portfolio weights for each date using historical data up to that date.

    Args:
        data (pd.DataFrame): DataFrame with dates as index and stocks as columns.
        lookback_window (int): Number of days to look back for the optimization.

    Returns:
        pd.DataFrame: A DataFrame with dates as index and stocks as columns, containing weights.
    """
    weights_list = []
    dates = []
    for date in data.index[lookback_window:]:
        window_data = data.loc[:date].tail(lookback_window)
        mu = expected_returns.mean_historical_return(window_data)
        S = risk_models.sample_cov(window_data)
        ef = EfficientFrontier(mu, S)
        try:
            ef.max_sharpe()
            clean_weights = ef.clean_weights()
        except Exception as e:
            # In case optimization fails, assign zero weights.
            clean_weights = {stock: 0 for stock in data.columns}
        weights_list.append(clean_weights)
        dates.append(date)
    weights_df = pd.DataFrame(weights_list, index=dates)
    return weights_df

In [ ]:
# Compute point-in-time fundamental ratios (requires stock data to be downloaded first)
hist_fundamentals = compute_historical_fundamentals(historical_fundamentals, "data/training_set.csv")
print(f"Historical fundamentals computed for {len(hist_fundamentals)} tickers")

# Preview one ticker
sample_ticker = list(hist_fundamentals.keys())[0]
print(f"\nSample ({sample_ticker}):")
hist_fundamentals[sample_ticker].tail()

In [ ]:
# Explore fundamentals distributions (latest available values per ticker)
latest_fund = pd.DataFrame({
    ticker: df.iloc[-1] for ticker, df in hist_fundamentals.items()
}).T

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, ["HistPE", "PB", "ROE", "ProfitMargin", "DebtToEquity", "FCFYield"]):
    if col not in latest_fund.columns:
        ax.set_title(f"{col} (not available)")
        continue
    data = latest_fund[col].dropna()
    if len(data) > 5:
        lo, hi = data.quantile(0.05), data.quantile(0.95)
        data = data[(data >= lo) & (data <= hi)]
    ax.hist(data, bins=30, edgecolor='black')
    ax.set_title(col)
    ax.set_xlabel("Value")
fig.suptitle("Historical Fundamentals Distributions (latest quarter, 5th-95th pct)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Feature engineering: Technical indicators, fundamentals
# SMA, RSI, MACD, etc. etc. basically, seeing which indicator sticks
def create_stock_features(stocks, stock_data_filename, hist_fund_dict=None) -> pd.DataFrame:
    feature_rows = []
    regime_df = classify_regimes(sp500)

    # Build ticker -> sector map once (used for SectorRank_4wk post-step)
    _sector_map = {}
    try:
        _sectors_df = pd.read_excel("data/100_stocks_sectors.xlsx")
        for sector_name in _sectors_df.columns:
            for t in _sectors_df[sector_name].dropna().astype(str).str.strip():
                _sector_map[t] = sector_name
    except Exception as e:
        print(f"WARN: could not load sector map ({e}); SectorRank_4wk will be NaN")

    # Macro feature renames: keep legacy column names for the original three FRED series.
    # Everything else passes through verbatim, so new FRED series auto-wire.
    FRED_RENAME = {
        "real_pce_yoy":  "Inflation",
        "FEDFUNDS":      "InterestRate",
        "UNRATE":        "UnemploymentRate",
    }

    for stock in stocks:
        try:
            prices = extract_ticker_dataframe(stock_data_filename, stock)
        except ValueError:
            continue
        features = pd.DataFrame(index=prices.index)

        sma = indicators.sma_strategy(prices["Close"].to_frame(), 5, 20)
        features["SMA_5v20"] = sma["signal_raw"]

        features["Volume"] = prices["Volume"]

        rsi = indicators.rsi_strategy(prices["Close"].to_frame())
        features["RSI"] = rsi["rsi"]

        macd = indicators.macd_strategy(prices["Close"].to_frame())
        features["MACD"] = macd["signal_raw"]

        bands = indicators.bollinger_strategy(prices["Close"].to_frame())
        features["Bollinger_Bands"] = bands["signal_ternary"]

        atr = indicators.atr_indicator(prices[["High", "Low", "Close"]])
        features["ATR"] = atr["signal"]

        stochastic = indicators.stochastic_strategy(prices[["High", "Low", "Close"]])
        features["Stochastic"] = stochastic["signal"]

        obv = indicators.obv_strategy(prices[["Close", "Volume"]])
        features["OBV"] = obv["obv"]

        adx = indicators.adx_strategy(prices[["High", "Low", "Close"]])
        features["ADX"] = adx["adx"]

        aroon = indicators.aroon_strategy(prices[["High", "Low"]])
        features["Aroon"] = aroon["aroon_oscillator"]

        # Return features (used for labels + sector rank input)
        features["Returns-3wk-1wklag"] = prices["Close"].pct_change(periods=3).shift(1)
        features["Returns-1wk-0wklag"] = prices["Close"].pct_change()

        # Macro features (loop dynamically so new FRED series auto-wire)
        fred_aligned = fred_data.reindex(prices.index, method="ffill")
        for col in fred_aligned.columns:
            features[FRED_RENAME.get(col, col)] = fred_aligned[col]

        # === FUNDAMENTALS (point-in-time, no look-ahead) ===
        if hist_fund_dict is not None and stock in hist_fund_dict:
            fund_df = hist_fund_dict[stock]
            fund_aligned = fund_df.reindex(features.index, method="ffill")
            for col in fund_aligned.columns:
                features[col] = fund_aligned[col]

        # Forward-return labels
        features["Returns-future-1wk"] = prices["Close"].pct_change(periods=1).shift(-1)
        features["Returns-future-2wk"] = prices["Close"].pct_change(periods=2).shift(-2)

        features["Bull_Probability"] = regime_df["Bull_Prob"]

        # Volatility features
        features["SP500-Log-Returns"] = sp500["Log_Returns"]
        features["SP500-Returns"] = sp500["Returns_6wk"]

        prices["Log_Returns"] = np.log(prices["Close"] / prices["Close"].shift(1))
        prices["Volatility_4wk"] = prices["Log_Returns"].rolling(window=4).std()
        prices["Volatility_12wk"] = prices["Log_Returns"].rolling(window=12).std()
        features["Volatility_Spike"] = prices["Volatility_4wk"] / prices["Volatility_12wk"]

        features["Beta_26wk"] = compute_beta(prices["Log_Returns"], sp500["Log_Returns"])

        features["Week-of-year"]     = prices.index.isocalendar().week
        features["Month-of-year"]    = prices.index.month
        features["Quarter"]          = prices.index.quarter
        features["Presidential-year"] = prices.index.to_series().apply(presidential_term_year)

        features["Stock"] = stock
        features = features.reset_index().rename(columns={"index": "Date"})
        feature_rows.append(features)

    features_long = pd.concat(feature_rows, ignore_index=True)

    # === Sector-relative strength rank (cross-sectional, after per-stock loop) ===
    features_long["_sector"] = features_long["Stock"].map(_sector_map)
    features_long = features_long.sort_values(["Stock", "Date"])
    features_long["_ret4wk"] = (
        features_long.groupby("Stock", group_keys=False)["Returns-1wk-0wklag"]
                     .rolling(4).sum().reset_index(level=0, drop=True)
    )
    features_long["SectorRank_4wk"] = (
        features_long.groupby(["Date", "_sector"])["_ret4wk"].rank(pct=True)
    )
    features_long = features_long.drop(columns=["_sector", "_ret4wk"])

    features_long = features_long.set_index("Date").dropna()
    return features_long

def presidential_term_year(date):
    election_years = [2000, 2004, 2008, 2012, 2016, 2020, 2024]
    year = date.year
    for start in reversed(election_years):
        if year >= start:
            return (year - start) + 1
    return np.nan

overall_df = create_stock_features(overall_tickers, "data/training_set.csv",
                                    hist_fund_dict=hist_fundamentals)


In [ ]:
def buy_low_sell_high_strategy(row):
    returns_prev_week = row['Returns-1wk-0wklag']
    if returns_prev_week < 0:
        return 1 # Buy
    else:
        return 0 # Sell

def buy_low_sell_high_model(df):
    df['Signal'] = df.apply(buy_low_sell_high_strategy, axis=1)
    df = df.sort_values(by='Date')
    return df

def always_buy_model_label(row):
    return 1

def always_buy_model(df):
    df['Signal'] = df.apply(always_buy_model_label, axis=1)
    df = df.sort_values(by='Date')
    return df

def buy_and_hold_strategy(row):
    if row['Date'] == pd.to_datetime("2000-01-01"):
        return 1 # Buy
    else:
        return 0 # Hold

#Will need to differentiate between models where 0 is hold and 0 is buy at some point.

def buy_and_hold_model(df):
    df['Signal'] = df.apply(buy_and_hold_strategy, axis=1)
    df = df.sort_values(by='Date')
    return df

def buy_low_sell_high(prices: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=prices.index)
    df["Price"] = prices
    df['Return'] = prices.pct_change()
    df['Signal'] = (df['Return'].shift(1) < 0).astype(int)
    df['Strategy'] = df['Signal'] * df['Return']
    df['Equity'] = (1 + df['Strategy']).cumprod()
    return df

In [ ]:
def backtest_strategy(model_func) -> pd.DataFrame:
    results = {}
    for ticker in stocks:
        try:
            prices_df = extract_ticker_dataframe("data/training_set.csv", ticker)
        except ValueError:
            continue
        results[ticker] = model_func(prices_df.dropna()["Close"])

    return results


curve = backtest_strategy(buy_low_sell_high)

In [ ]:
def label_signal2(row):
    r1 = row['Returns-future-1wk']
    r2 = row['Returns-future-2wk']
    if r1 < -0.01 and r2 < -0.015:
        return 0  # Sell
    elif r1 > 0.01 and r2 > 0.015:
        return 1  # Buy
    else:
        return 2  # Hold


def train_val_test_split(df):
    df['Signal'] = df.apply(label_signal2, axis=1)
    df = df.dropna(subset=['Returns-future-1wk', 'Returns-future-2wk', 'Signal']) # dropping rows with no signal

    # Sort by date?
    df = df.sort_values(by='Date')

    # 70% train, 15% val, 15% test
    split_1 = int(len(df) * 0.7)
    split_2 = int(len(df) * 0.85)

    train = df.iloc[:split_1]
    val = df.iloc[split_1:split_2]
    test = df.iloc[split_2:]

    return train, val, test


In [ ]:
def train_model(sets: tuple):
    train = sets[0]
    features = train.drop(['Stock', 'Signal', 'Returns-future-1wk', 'Returns-future-2wk'], axis=1).columns.values
    X_train = train[features]
    y_train = train['Signal']

    # balanced class weights to balance any over-representation
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, class_weights))
    print("Class Weights:", class_weight_dict)

    sample_weights = y_train.map(class_weight_dict)

    # Train the model
    model = XGBClassifier(
        objective='multi:softprob',  # for multi-class
        num_class=len(classes),
        eval_metric='mlogloss',
        tree_method='hist',
        subsample=0.6,
        colsample_bytree=0.9,
        reg_lambda=5,
        reg_alpha=5,
        booster='gbtree',
        learning_rate=0.01,
        gamma=0.5,
        min_child_weight=5,
        n_estimators=200,
        use_label_encoder=False,
        random_state=42,
        max_depth = 12,
    )
    model.fit(X_train, y_train, sample_weight=sample_weights)

    return model

def predict(model: XGBClassifier, sets: tuple, val_or_test: str):
    val = sets[1]
    test = sets[2]
    features = val.drop(['Stock', 'Signal', 'Returns-future-1wk', 'Returns-future-2wk'], axis=1).columns.values

    if val_or_test == 'val':
        x = val[features]
        y = val['Signal']
    else:
        x = test[features]
        y = test['Signal']

    y_pred = model.predict(x)
    return classification_report(y, y_pred, output_dict=True)



In [ ]:
overall_sets = train_val_test_split(overall_df)

overall_model = train_model(overall_sets)

In [ ]:
predict_overall = predict(overall_model, overall_sets, val_or_test='val')

In [ ]:
predict_overall

In [ ]:
importance2 = overall_model.get_booster().get_score(importance_type='gain')

print(importance2)

In [ ]:
# Implementing with a real portfolio
real_stock_portfolio_1 = [
    "AMD", "NVDA", "RDDT", "SMCI"
]

real_stock_portfolio_2 = [
    "CRDO", "GOOG", "APP", "ASAN", "CLS", "META", "QQQ", "LUNR", "JPM", "PANW", "HOOD", "SPY", "SKYW", "TSM", "UBER", "VRT", "WMT", "NLR", "URA", "BITO", "COST", "XYZ"
]

real_stock_portfolio_3 = [
    "AAPL", "ABNB", "ACN", "ALAB", "AMD", "AMZN", "ANET", "AOSL", "APP",
    "ASAN", "ASML", "AVGO", "BAH", "BITO", "BWXT", "CLS", "COHR", "COIN", "COST",
    "COWG", "CPRX", "CRDO", "CRM", "CRWV", "DAVE", "DELL", "DKNG", "DOCS",
    "DXPE", "EPD", "FBTC", "FVRR", "GOOG", "GRNY", "HOOD", "IHAK", "INTA", "IONQ",
    "JPM", "LITE", "LQDT", "LUNR", "META", "MRVL", "MSFT", "MU", "NBIS", "NEE",
    "NFLX", "NLR", "NNE", "NUTX", "NVDA", "NVDY", "NVO", "OUST", "OXY", "PANW",
    "PEP", "PLD", "PLTR", "PYPL", "QCOM", "QTUM", "RBRK", "RDDT", "RDNT", "REAL",
    "RGTI", "S", "SAIC", "SCHD", "SEZL", "SKYW", "SMCI", "SMTC", "SNOW", "SOXL",
    "SYM", "TEAM", "TEM", "TOST", "TSM", "U", "UBER", "UPST", "URA", "VIST",
    "VRT", "WMT", "WRD", "XYZ", "HIMS", "OSCR"
]

In [ ]:
def run_model(stock_portfolio):
    # Part 1: Getting recommendations
    real_df = create_stock_features(stock_portfolio, "data/stocks.csv",
                                       hist_fund_dict=hist_fundamentals)

    real_df = real_df.sort_values(by='Date')
    today_stocks_features = real_df.loc[[real_df.index.max()]]

    features = real_df.drop(['Stock', 'Returns-future-1wk', 'Returns-future-2wk'], axis=1).columns.values
    model = overall_model

    todays_pred = model.predict(today_stocks_features[features])
    todays_probs = model.predict_proba(today_stocks_features[features])  # shape: (n_samples, 3)
    stock_col = today_stocks_features["Stock"].reset_index().drop(columns="Date")
    pred_col = pd.DataFrame(todays_pred)
    probs_col = pd.DataFrame(todays_probs)

    # Making recommendations per stock
    recommendations = stock_col.join(pred_col)
    recommendations.columns = ["Stock", "Recommendation"]
    recommendations["Recommendation"] = recommendations["Recommendation"].map({0: 'Sell', 1: 'Buy', 2: 'Hold'})

    return recommendations, 0

(recommendations, weights) = run_model(real_stock_portfolio_3) # <- Replace with the portfolio you want here


In [ ]:
#recommendations.sort_values("Recommendation", ascending=False, inplace=True)
recommendations.to_csv("recommendations.csv")

In [ ]:
# ============================================================
# Archived Experiments (SHAP, Hyperparameter Search, Markowitz)
# ============================================================
# Uncomment sections below to run them.

# --- SHAP Analysis ---
# import shap
# booster = overall_model.get_booster()
# explainer = shap.TreeExplainer(booster)
# shap_values = explainer.shap_values(X_test)
# shap.initjs()
# shap.summary_plot(shap_values, X_test, plot_type="bar")
# shap.summary_plot(shap_values[:, :, 0], X_test)  # buy
# shap.summary_plot(shap_values[:, :, 1], X_test)  # hold
# shap.summary_plot(shap_values[:, :, 2], X_test)  # sell
# shap.dependence_plot("Bull_Probability", shap_values[:, :, 2], X_test)
# shap.dependence_plot("ATR", shap_values[:, :, 2], X_test)
# shap.dependence_plot("MACD", shap_values[:, :, 2], X_test)

# --- Hyperparameter Search ---
# from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
# param_dist = {
#     'n_estimators': [100, 200, 300, 400, 500, 700],
#     'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2],
#     'max_depth': [3, 4, 5, 6, 8, 10, 12],
#     'min_child_weight': [1, 3, 5, 7, 10],
#     'gamma': [0, 0.1, 0.25, 0.5, 1],
#     'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
#     'reg_alpha': [0, 0.1, 0.5, 1, 5, 10],
#     'reg_lambda': [0, 0.1, 1, 5, 10, 20],
#     'booster': ['gbtree'],
#     'tree_method': ['hist'],
#     'objective': ['multi:softprob'],
#     'num_class': [3],
# }
# xgb_clf = XGBClassifier(use_label_encoder=False, random_state=42, verbosity=0)
# search = RandomizedSearchCV(
#     estimator=xgb_clf, param_distributions=param_dist,
#     n_iter=75, scoring='f1_macro', cv=3, verbose=2, n_jobs=-1, random_state=42
# )
# search.fit(X_train, y_train, sample_weight=sample_weights)
# print("Best Params:", search.best_params_)

# --- Markowitz Mean-Variance Portfolio ---
# buy_recommendations = recommendations[recommendations.Recommendation == "Buy"].drop(columns="Recommendation")
# rec_array = buy_recommendations.to_numpy().flatten().tolist()
# buy_stocks_history = extract_ticker_dataframe("stocks.csv", rec_array[0])["Close"]
# for i in range(1, len(rec_array)):
#     a = extract_ticker_dataframe("stocks.csv", rec_array[i])["Close"]
#     buy_stocks_history = pd.concat([buy_stocks_history, a], axis=1, join='inner')
# buy_stocks_history.columns = rec_array
#
# def compute_adjusted_mu(buy_probs, baseline_mu, alpha=0.01):
#     tickers = baseline_mu.index.intersection(buy_probs.index)
#     adjusted_mu = baseline_mu.loc[tickers] * (1 + alpha * (buy_probs.loc[tickers] - 0.5))
#     return adjusted_mu
#
# probs = model.predict_proba(today_stocks_features[features])
# probs_db = pd.DataFrame(probs, columns=["Hold", "Buy", "Sell"], index=today_stocks_features["Stock"].values)
# buy_probs = probs_db["Buy"]
# baseline_mu = expected_returns.mean_historical_return(buy_stocks_history)
# adjusted_mu = compute_adjusted_mu(buy_probs, baseline_mu, alpha=0.05)
# S = risk_models.sample_cov(buy_stocks_history)
# common_tickers = adjusted_mu.index.intersection(S.index)
# S = S.loc[common_tickers, common_tickers]
# ef = EfficientFrontier(adjusted_mu, S)
# ef.max_sharpe()
# clean_weights = ef.clean_weights()
# weights = OrderedDict()
# for key, value in clean_weights.items():
#     if value != 0.0:
#         weights[key] = value
# print(weights)
